# Lab 16: Amazon S3 (boto3)

A bucket's life: create it, keep it private, open it with an ACL and then a bucket policy, and recover deleted files via versioning.

Run `make up LAB=06-s3` first.

This lab involves access and Floci only does deny it when enforcement settings are on.

Run `make start LAB=06-s3` second for the starting state.

## Setup

Settings come from `.env`. Checking target to know wher ecalls go.

In [ ]:
import os
import io
import time
import urllib.error
import urllib.parse
import urllib.request
import json

import boto3
from botocore.exceptions import ClientError
from dotenv import find_dotenv, load_dotenv

from PIL import Image, ImageDraw

load_dotenv(find_dotenv(), override=True)
ENDPOINT = os.environ.get("AWS_ENDPOINT_URL", "real AWS")
print("Target: ", ENDPOINT or "real AWS")

REGION = boto3.session.Session().region_name
s3  = boto3.client('s3')
iam = boto3.client('iam')
sts = boto3.client('sts')

ACCOUNT = sts.get_caller_identity()['Account']
BUCKET  = f"reportbucket-{ACCOUNT}"
ROLE    = "EC2InstanceProfileRole" 

### Helpers

In [ ]:
def anonymous_get(url):
    try:
        with urllib.request.urlopen(url) as response:
            return response.status, response.read()
    except urllib.error.HTTPError as error:
        return error.code, b""

def object_url(key, **query):
    AWS_BASE = f"https://{BUCKET}.s3.{REGION}.amazonaws.com"
    base = f"{ENDPOINT}/{BUCKET}"if ENDPOINT else AWS_BASE

    return f"{base}/{key}" + (f"?{urllib.parse.urlencode(query)}" if query else "")

def wait_until(check, what, timeout=30):
    deadline = time.time() + timeout
    while not check():
        assert time.time() < deadline, f"timed out waiting for: {what}"
        time.sleep(2)

def error_code(call, **kwargs):
    """S3 error code a call fails with; or None if it succeeds."""
    try:
        call(**kwargs)
    except ClientError as error:
        return error.response["Error"]["Code"]

def delete_bucket(name):
    try:
        object_response_paginator = s3.get_paginator('list_object_versions')
        for obj_resp_iter in object_response_paginator.paginate(Bucket=BUCKET):
            for item in obj_resp_iter.get("Versions", []) + obj_resp_iter.get("DeleteMarkers", []):
                s3.delete_object(Bucket=BUCKET, Key=item['Key'], VersionId=item['VersionId'])
        s3.delete_bucket(Bucket=name)
    except s3.exceptions.NoSuchBucket:
        pass

Removing any bucket left by an earlier run:

In [ ]:
delete_bucket(BUCKET)

## Task 1: Create a bucket
The lab turns ACLs on by choosing the *Object writer* ownership setting. Buckets on AWS block all public access by default, so we set that explicitly and every target starts the same.

In [ ]:
BLOCK_ALL = dict.fromkeys(
    ['BlockPublicAcls', 'IgnorePublicAcls',
    'BlockPublicPolicy', 'RestrictPublicBuckets'],
    True
)

create_args = {
    'Bucket':BUCKET,
    'ObjectOwnership': 'ObjectWriter'
}

if REGION != 'us-east-1':
    create_args['CreateBucketConfiguration'] = {
        'LocationConstraint': REGION
    }

s3.create_bucket(**create_args)

s3.put_public_access_block(
    Bucket=BUCKET,
    PublicAccessBlockConfiguration=BLOCK_ALL,
)

rules = s3.get_bucket_ownership_controls(Bucket=BUCKET)['OwnershipControls']['Rules']
assert rules[0]['ObjectOwnership'] == 'ObjectWriter'

## Task 2: Upload an object to the bucket
We draw our own chart here and upload it

In [ ]:
def make_report():
    image = Image.new("RGB", (640, 360), "white")
    draw = ImageDraw.Draw(image)
    for i, height in enumerate((69, 200, 160, 42, 67)):
        left = 60 + i * 110
        draw.rectangle((left, 320 - height, left + 80, 320), fill=(40, 90, 160))
    out = io.BytesIO()
    image.save(out, format="PNG")
    return out.getvalue()

FILE_KEY = "new-report.png"

s3.put_object(
    Bucket=BUCKET, Key=FILE_KEY,
    Body=make_report(), ContentType="image/png"
)

keys = [obj["Key"] for obj in s3.list_objects_v2(Bucket=BUCKET)["Contents"]]
assert keys == [FILE_KEY]

## Task 3: Make an object public
Objects are private by default, so an anonymous request is refused. Block Public Access outranks the object's ACL, so asking for a public ACL fails until the block is lifted.

In [ ]:
report_url = object_url(FILE_KEY)

wait_until(
    lambda: anonymous_get(report_url)[0] == 403,
    "a private object to answer 403 (on Floci, start it with: make up LAB=06-s3)",
)

# Block Public Access wins over the ACL
assert error_code(s3.put_object_acl, Bucket=BUCKET,
        Key=FILE_KEY, ACL="public-read") == "AccessDenied"

In [ ]:
s3.put_public_access_block(
    Bucket=BUCKET,
    PublicAccessBlockConfiguration=dict.fromkeys(BLOCK_ALL, False),
)
s3.put_object_acl(Bucket=BUCKET, Key=FILE_KEY, ACL="public-read")

wait_until(
    lambda: anonymous_get(report_url)[0] == 200,
    "the public object to be readable"
)

assert anonymous_get(report_url)[1].startswith(b"\x89PNG")

## Task 4: Test connectivity from the EC2 instance
The lab opens a Session Manager shell on a bastion host. \
Here, neither the shell nor the instance exists, so the notebook assumes the instance's role instead and makes the same calls with its credentials.

The role can list and read but not write, so the upload is refused.

In [ ]:
role_arn = iam.get_role(RoleName=ROLE)["Role"]["Arn"]
creds = sts.assume_role(RoleArn=role_arn, RoleSessionName="bastion-host")["Credentials"]
bastion = boto3.Session(
    aws_access_key_id=creds["AccessKeyId"],
    aws_secret_access_key=creds["SecretAccessKey"],
    aws_session_token=creds["SessionToken"],
).client("s3")

assert BUCKET in [b["Name"] for b in bastion.list_buckets()["Buckets"]]
keys = [obj["Key"] for obj in bastion.list_objects_v2(Bucket=BUCKET)["Contents"]]
assert keys == [FILE_KEY]

REPORT_FILE_KEY = "report-test1.txt"
REPORT_FILE_CONTENT = b"first report\n"

denied = error_code(
    bastion.put_object, Bucket=BUCKET,
    Key=REPORT_FILE_KEY, Body=REPORT_FILE_CONTENT
)
assert denied == "AccessDenied"

## Task 5: Create a bucket policy
First we add another private object, then then a bucket policy that lets the instance's role read and write \
You can build the policy via the [AWS Policy Generator](https://awspolicygen.s3.amazonaws.com/policygen.html), which produces the JSON used blow.

In [ ]:
SAMPLE_KEY = "sample-file.txt"
SAMPLE_CONTENT = b"Nothing but a sample in this world.\n"
s3.put_object(
    Bucket=BUCKET, Key=SAMPLE_KEY,
    Body=SAMPLE_CONTENT
)

sample_url = object_url(SAMPLE_KEY)
wait_until(
    lambda: anonymous_get(sample_url)[0] == 403,
    "the new object to be private"
)

Notice below how for resource we mention `BUCKET_ARN/*` to point at the objects, not the bucket itself.

In [ ]:
bucket_arn = f"arn:aws:s3:::{BUCKET}"
policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "InstanceReadWrite",
            "Effect": "Allow",
            "Principal": {"AWS": role_arn},
            "Action": ["s3:GetObject", "s3:PutObject"],
            "Resource": f"{bucket_arn}/*",
        }
    ],
}
s3.put_bucket_policy(Bucket=BUCKET, Policy=json.dumps(policy))

wait_until(
    lambda: error_code(
        bastion.put_object, Bucket=BUCKET,
        Key=REPORT_FILE_KEY, Body=REPORT_FILE_CONTENT
    ) is None,
    "the instance's upload to be allowed"
)
assert bastion.get_object(Bucket=BUCKET, Key=SAMPLE_KEY)['Body'].read() == SAMPLE_CONTENT

assert anonymous_get(sample_url)[0] == 403

Now, let's give the  public read permission:

In [ ]:
policy["Statement"].append(
    {
        "Sid": "PublicRead",
        "Effect": "Allow",
        "Principal": "*",
        "Action": "s3:GetObject",
        "Resource": f"{bucket_arn}/*",
    }
)
s3.put_bucket_policy(Bucket=BUCKET, Policy=json.dumps(policy))

wait_until(
    lambda: anonymous_get(sample_url)[0] == 200,
    "the public read policy to apply"
)
assert anonymous_get(sample_url)[1] == SAMPLE_CONTENT

## Task 6: Explore versioning
We enable versioning on the whole bucket.

Given that `smaple-file.txt` was uploaded beforehand, its first version has ID `null`.

In [ ]:
s3.put_bucket_versioning(
    Bucket=BUCKET,
    VersioningConfiguration={"Status":"Enabled"}
)

SAMPLE_UPDATED_CONTENT = b"There is more to this sample.\nWho knew...\n"
s3.put_object(
    Bucket=BUCKET,
    Key=SAMPLE_KEY, Body=SAMPLE_UPDATED_CONTENT
)


def versions_of(key):
    listing = s3.list_object_versions(Bucket=BUCKET, Prefix=key)
    return listing.get("Versions", []), listing.get("DeleteMarkers", [])

versions, _ = versions_of(SAMPLE_KEY)
latest      = next(v for v in versions if v["IsLatest"])
original    = next(v for v in versions if not v["IsLatest"])
assert original['VersionId'] == 'null'

# We get the latest version, unless we name which version
assert s3.get_object(Bucket=BUCKET, Key=SAMPLE_KEY)['Body'].read() == SAMPLE_UPDATED_CONTENT
old_content = s3.get_object(Bucket=BUCKET, Key=SAMPLE_KEY, VersionId='null')['Body'].read()
assert old_content == SAMPLE_CONTENT

# Public statement allows s3:GetObject only, not s3:GetObjectVersion
assert anonymous_get(object_url(SAMPLE_KEY, versionId='null'))[0] == 403

Deleting a versioned object only adds a delete marker.

In [ ]:
s3.delete_object(Bucket=BUCKET, Key=SAMPLE_KEY)
assert error_code(
    s3.get_object,
    Bucket=BUCKET, Key=SAMPLE_KEY
) == "NoSuchKey"

versions, markers = versions_of(SAMPLE_KEY)
assert len(versions) == 2 and markers[0]['IsLatest']

Just by deleting the marker, we get our sample file back.

In [ ]:
s3.delete_object(Bucket=BUCKET, Key=SAMPLE_KEY, VersionId=markers[0]['VersionId'])
assert s3.get_object(
    Bucket=BUCKET, Key=SAMPLE_KEY,
)['Body'].read() == SAMPLE_UPDATED_CONTENT

Targeting a specific version for deletion removes it for good with no marker left, which brings back the older text here:

In [ ]:
s3.delete_object(
    Bucket=BUCKET,
    Key=SAMPLE_KEY, VersionId=latest['VersionId']
)

versions, markers = versions_of(SAMPLE_KEY)
assert len(versions) == 1 and not markers

wait_until(
    lambda: anonymous_get(sample_url)[1] == SAMPLE_CONTENT,
    "the original text to be public again"
)

## Clean up

We delete the bucket+versions, so `make end` can remove the starting state.

In [ ]:
delete_bucket(BUCKET)

Now run `make end LAB=06-s3`.